# 🔎 VeriFile AI — Automated Payment Receipt Forensics & Integrity Triage

**Single & Multi-Upload Payment Receipt Forensic System**  
Upload payment receipt images (JPG, PNG, WebP) — the system automatically detects whether the receipt is authentic, exhibits traces of **Generative AI (C2PA / SynthID)**, **graphic editing software (Photoshop/Canva)**, **screen recapture / physical phone screen photographs**, and extracts structured receipt details while verifying **internal arithmetic amount consistency** automatically.

**How to use in Google Colab:**
1. Run the cells in sequential order.
2. The core forensic engine runs 100% locally without requiring a GPU.
3. (Optional GPU) Select *Runtime > Change runtime type > T4 GPU* if you wish to run Gemma 4 Multimodal.

In [ ]:
import os, sys, shutil, subprocess
print("Python:", sys.version.split()[0])
try:
    import torch
    print("GPU active:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Device:", torch.cuda.get_device_name(0))
except ImportError:
    print("Torch not installed; core forensic engine runs in full CPU mode.")

## 1. Clone VeriFile-AI Repository
Fetch the latest source code directly from the GitHub repository.

In [ ]:
!git clone https://github.com/alifakbxr/VeriFile-AI.git /content/VeriFile-AI 2>/dev/null || (cd /content/VeriFile-AI && git pull)
import os, sys
if os.path.exists("/content/VeriFile-AI"):
    os.chdir("/content/VeriFile-AI")
    if "/content/VeriFile-AI" not in sys.path:
        sys.path.insert(0, "/content/VeriFile-AI")
print("Working directory:", os.getcwd())

## 2. Install Dependencies, RapidOCR, and Run Test Suite
Installs `requirements-ocr.txt` (Pillow, OpenCV, PyMuPDF, defusedxml, RapidOCR, ONNX Runtime) and executes all 48 automated unit & security tests.

In [ ]:
%pip -q install -r requirements-ocr.txt
!python demo/create_samples.py
!PYTHONPATH=. pytest -v

## 3. Demo: Single-Upload Triage of Authentic vs AI-Generated Transfer Receipt
Test automated triage directly on sample payment receipt fixtures (`demo/TEST DATA/`):

In [ ]:
from verifile.forensics import analyze_file
from pathlib import Path

orig_path = Path("demo/TEST DATA/202610300859170551-1793419781221.jpg")
fake_path = Path("demo/TEST DATA/202610300859170551-179341978122122.jpg")

print("=== 1. TEST AUTHENTIC RECEIPT ===")
orig_rep, _, _ = analyze_file(orig_path)
print(f"Verdict: {orig_rep.get('verdict_title')}")
print(f"Category: {orig_rep.get('verdict_category')} (Risk: {orig_rep.get('risk_level')})")
print(f"Bank: {orig_rep.get('receipt_data', {}).get('bank')}")
print(f"Amount: {orig_rep.get('receipt_data', {}).get('amounts')}")

print("\n=== 2. TEST GENERATIVE AI MODIFIED RECEIPT (FAKE) ===")
fake_rep, _, _ = analyze_file(fake_path)
print(f"Verdict: {fake_rep.get('verdict_title')}")
print(f"Category: {fake_rep.get('verdict_category')} (Risk: {fake_rep.get('risk_level')})")
print(f"Bank: {fake_rep.get('receipt_data', {}).get('bank')}")
print(f"Amount: {fake_rep.get('receipt_data', {}).get('amounts')}")
print("Forensic Findings:")
for f in fake_rep.get('forensics', {}).get('findings', []):
    if f.get('severity') in ['alert', 'review']:
        print(f" - [{f['severity'].upper()}] {f['title']}: {f['detail']}")

## 4. (Optional) Enable Gemma 4 Multimodal (GPU) and/or Gemini API Fallback (No GPU)

- **Primary (GPU):** Default model `google/gemma-4-E2B-it` requires *T4 GPU* and Hugging Face access.
- **Automated Fallback (CPU/GPU):** Store `GEMINI_API_KEY` in Colab **Secrets** (key icon in the left panel). When GPU is unavailable or local Gemma fails to load, the system seamlessly falls back to **Gemini API** (`gemini-2.5-flash`) without requiring a GPU or large model download.

In [ ]:
# 1) Configure Gemini API fallback from Colab Secrets (runs on CPU or GPU).
has_gemini_fallback = False
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
    if gemini_key:
        get_ipython().system('pip install -q -r requirements-gemini.txt')
        os.environ['GEMINI_API_KEY'] = gemini_key
        os.environ['GEMINI_FALLBACK'] = '1'
        os.environ.setdefault('GEMINI_MODEL_ID', 'gemini-2.5-flash')
        del gemini_key
        has_gemini_fallback = True
        print('✅ Gemini API fallback active (Colab Secret GEMINI_API_KEY detected).')
except Exception as e:
    print('ℹ️ GEMINI_API_KEY not set in Colab Secrets (optional for GPU-less AI fallback):', type(e).__name__)

# 2) Set True only if you want to load local Gemma 4 on Colab GPU.
ENABLE_GEMMA = False
if ENABLE_GEMMA:
    import torch
    if not torch.cuda.is_available():
        if has_gemini_fallback:
            print('GPU not active; when the AI checkbox is enabled in the UI, the app will automatically use Gemini API fallback.')
        else:
            raise RuntimeError('Select T4 GPU in Runtime > Change runtime type, or configure Colab Secret GEMINI_API_KEY for fallback.')
    else:
        # Ensure bitsandbytes >= 0.46.1 and accelerate are installed for 4-bit quantization
        get_ipython().system('pip install -q -U "bitsandbytes>=0.46.1" "accelerate>=1.1.0" transformers')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-r', 'requirements-gemma.txt'], check=True)
        os.environ['GEMMA_4BIT'] = '1'
        os.environ['GEMMA_MODEL_ID'] = 'google/gemma-4-E2B-it'
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
            if hf_token:
                from huggingface_hub import login
                login(token=hf_token, add_to_git_credential=False)
                del hf_token
                print('Hugging Face authentication configured.')
        except Exception as e:
            print('Colab secret HF_TOKEN not set:', type(e).__name__)
        print('Gemma is ready for use when the AI checkbox is enabled in the UI.')
else:
    if has_gemini_fallback:
        print('Local Gemma skipped; AI checkbox in UI will use Gemini API fallback without GPU.')
    else:
        print('Skipping Gemma & Gemini. Core forensic engine runs in full CPU mode.')

## 5. Launch Gradio Demo (Supports Multi-Upload Batch Receipt Triage)

Launches an interactive web interface supporting **multi-upload payment receipt triage**. Upload one or multiple receipt images — the system immediately detects Generative AI artifacts (C2PA/SynthID), graphic editing software, physical screen photos, and extracts structured receipt details for each file automatically without needing manual selection of original vs fake reference files.


In [ ]:
SHARE_PUBLIC_DEMO = False

import os
auth = None
if os.getenv("VERIFILE_AUTH_USER") and os.getenv("VERIFILE_AUTH_PASS"):
    auth = (os.environ["VERIFILE_AUTH_USER"], os.environ["VERIFILE_AUTH_PASS"])

import app as verifile_app
verifile_app.build_app().launch(
    server_name="0.0.0.0",
    share=SHARE_PUBLIC_DEMO,
    auth=auth,
    debug=True,
    max_file_size="20mb"
)